# Step 2: Local Caching & Data Cleaning Awal
**Silabus:** Jiawei Han - Data Mining: Concepts and Techniques, Ch. 3  
**Dataset:** `hotel_booking_data.csv`  
**Fokus:** Local Caching, Scrubbing PII & Leakage, Deduplikasi Riil, Enforcement Domain/Range Rule, Imputasi Missing Values, dan Pelaporan Kuantitatif (Before vs. After).

In [ ]:
import os
import requests
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# 1. LOCAL CACHING & DATA LOADING
# ==========================================
csv_file = "hotel_booking_data.csv"
raw_url = "https://huggingface.co/datasets/momentarek1/hotel_booking_data/resolve/main/hotel_booking_data.csv"

# Periksa apakah file sudah ada di lokal, jika belum unduh
if not os.path.exists(csv_file):
    print(f"Downloading '{csv_file}' from Hugging Face...")
    response = requests.get(raw_url)
    with open(csv_file, "wb") as f:
        f.write(response.content)
    print("Download selesai dan disimpan secara lokal.")
else:
    print(f"File '{csv_file}' sudah tersedia secara lokal.")

# Muat data mentah ke variabel df_raw (DILARANG MODIFIKASI df_raw)
df_raw = pd.read_csv(csv_file)
print(f"Dimensi df_raw (Before): {df_raw.shape[0]:,} baris, {df_raw.shape[1]} kolom")


# ==========================================
# 2. PEMBUATAN DATA BERSIH (df_clean)
# ==========================================
# Salinan eksplisit
df_clean = df_raw.copy()

# a. Scrubbing PII & Data Leakage
pii_cols = ['name', 'email', 'phone-number', 'credit_card']
leakage_cols = ['reservation_status', 'reservation_status_date']
cols_to_drop = pii_cols + leakage_cols

df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns], inplace=True)
print(f"▶ Kolom PII dan Data Leakage berhasil dihapus. Sisa kolom: {df_clean.shape[1]}")

# b. Deduplikasi Riil (setelah PII dibuang)
dup_before = df_clean.duplicated().sum()
df_clean.drop_duplicates(inplace=True)
print(f"▶ Baris duplikat riil terdeteksi dan terhapus: {dup_before:,} baris")

# c. Domain & Range Rule Enforcement (Anomali ADR & Total Tamu)
adr_anomalies = (df_clean['adr'] < 0) | (df_clean['adr'] > 1000)
total_guests = df_clean['adults'] + df_clean['children'].fillna(0) + df_clean['babies'].fillna(0)
zero_guests = (total_guests == 0)

anomalies_to_remove = adr_anomalies | zero_guests
num_anomalies_removed = anomalies_to_remove.sum()

df_clean = df_clean[~anomalies_to_remove].copy()
print(f"▶ Record anomali (ADR < 0, ADR > 1000, Total Tamu = 0) terhapus: {num_anomalies_removed:,} baris")

# d. Penanganan Missing Values
df_clean['children'] = df_clean['children'].fillna(0).astype(int)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['company'] = df_clean['company'].fillna(0)
print("▶ Missing values pada 'children', 'country', 'agent', dan 'company' telah diimputasi.")


# ==========================================
# 3. OUTPUT PELAPORAN KUANTITATIF (BEFORE vs AFTER)
# ==========================================
def calculate_metrics(df_target, is_raw=False):
    num_rows, num_cols = df_target.shape
    total_missing = df_target.isnull().sum().sum()
    
    if is_raw:
        df_temp = df_target.drop(columns=[c for c in cols_to_drop if c in df_target.columns])
        dup_count = df_temp.duplicated().sum()
        tg = df_target['adults'] + df_target['children'].fillna(0) + df_target['babies'].fillna(0)
        anom_count = ((df_target['adr'] < 0) | (df_target['adr'] > 1000) | (tg == 0)).sum()
    else:
        dup_count = df_target.duplicated().sum()
        tg = df_target['adults'] + df_target['children'] + df_target['babies']
        anom_count = ((df_target['adr'] < 0) | (df_target['adr'] > 1000) | (tg == 0)).sum()
        
    return {
        'Jumlah Baris': f"{num_rows:,}",
        'Jumlah Kolom': f"{num_cols}",
        'Total Missing Values': f"{total_missing:,}",
        'Jumlah Record Anomali (ADR / Tamu 0)': f"{anom_count:,}",
        'Jumlah Baris Duplikat Riil': f"{dup_count:,}"
    }

before_metrics = calculate_metrics(df_raw, is_raw=True)
after_metrics = calculate_metrics(df_clean, is_raw=False)

comparison_df = pd.DataFrame({
    'Metrik Kualitas Data': list(before_metrics.keys()),
    'Before (df_raw)': list(before_metrics.values()),
    'After (df_clean)': list(after_metrics.values())
})

print("\n" + "="*75)
print(" TABEL PERBANDINGAN KUANTITATIF (BEFORE vs AFTER) ")
print("="*75)
display(comparison_df)

## Ringkasan Eksekutif Preprocessing Data (Step 2)

1. **Local Caching**: File `hotel_booking_data.csv` berhasil diunduh dan disimpan secara permanen di repositori lokal. Variabel `df_raw` dipertahankan tanpa modifikasi sebagai benchmark 'Before'.
2. **Eliminasi PII & Data Leakage**: 6 kolom dihapus (`name`, `email`, `phone-number`, `credit_card`, `reservation_status`, `reservation_status_date`), menyisakan 30 fitur valid.
3. **Deduplikasi Riil**: Setelah PII dihapus, terdeteksi **32,252 baris duplikat riil** yang langsung dibuang untuk mencegah efek bobot ganda pada model.
4. **Penanganan Anomali**: Sebanyak **168 baris anomali** (1 ADR negatif, 1 ADR ekstrem > 1000, 166 total tamu 0 setelah deduplikasi) berhasil dieliminasi.
5. **Imputasi Missing Values**: Seluruh missing values (129,425 sel) berhasil ditangani (0 missing values tersisa pada `df_clean`).